# Clustering: agrupar sem rótulo

**Para quem está começando.** Nenhuma linha de código aqui assume que você já viu agrupamento. A ideia é rodar de cima para baixo, célula por célula, e olhar os números.

O que vamos fazer, em ordem:

1. entender o que muda quando não existe rótulo para conferir;
2. ajustar o K-Means em dois grupos convexos e escolher o `k`;
3. ver o K-Means perder e o DBSCAN ganhar em duas meias-luas;
4. medir o resultado num dado de verdade, onde a verdade escondida só serve para conferir.

Roda no Google Colab ou no Jupyter do seu computador. Só precisa de `numpy`, `pandas`, `scikit-learn` e `matplotlib`.


## O que muda quando não existe rótulo

Nos outros notebooks havia um `y` para comparar. Aqui não há: o modelo recebe só as colunas e devolve os grupos. Isso muda tudo na avaliação, porque não existe acurácia.

O que existe são medidas internas da própria partição:

- **inércia**: soma das distâncias de cada ponto ao centro do seu grupo (só para K-Means). Quanto menor, mais compacto, mas ela sempre melhora quando se aumenta o número de grupos;
- **silhouette**: para cada ponto, compara a distância média dentro do seu grupo com a distância ao grupo vizinho mais próximo, e resume isso num número entre -1 e 1. Vale para qualquer partição, inclusive as que o K-Means não produz;
- **verdade escondida**: quando existe, serve para **conferir** o agrupamento com o índice de Rand ajustado (ARI). Nunca para escolher o número de grupos, senão o resultado deixa de ser não supervisionado.

E os dois algoritmos deste notebook atacam o problema de formas diferentes. O **K-Means** fixa o número de grupos e procura centros, então a fronteira entre grupos é sempre uma reta e os grupos saem convexos. O **DBSCAN** não fixa número nenhum: ele procura regiões densas e marca o que fica isolado como ruído (rótulo -1), então aceita qualquer formato, ao custo de dois parâmetros sensíveis.


## Passo 0: ambiente e configuração

Nada para instalar além do que já vem no Colab. A semente é a mesma em todas as células.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN, KMeans
from sklearn.datasets import load_digits, make_blobs, make_moons
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
plt.rcParams["figure.figsize"] = (9, 4)
print(f"✅ numpy {np.__version__} | semente {RANDOM_STATE}")

## Passo 1: dois grupos convexos, e a escolha do `k`

O primeiro conjunto é o caso fácil: quatro grupos redondos, bem separados. Aqui o K-Means é o modelo certo, e a pergunta é só quantos grupos.

O truque didático é olhar as duas colunas da tabela. A `inercia` cai sempre quando o `k` cresce, porque mais centros sempre aproximam os pontos deles, e com `k` igual ao número de pontos a inércia é zero. Ela sozinha não serve para decidir. O `silhouette` tem máximo, e é ele que aponta o `k`.


In [ ]:
X_blobs, y_blobs = make_blobs(n_samples=600, centers=4, cluster_std=1.0, random_state=RANDOM_STATE)
X_blobs = StandardScaler().fit_transform(X_blobs)

linhas = []
for k in range(2, 8):
    rotulos = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit_predict(X_blobs)
    linhas.append({"k": k, "inercia": KMeans(n_clusters=k, n_init=10,
                                              random_state=RANDOM_STATE).fit(X_blobs).inertia_,
                   "silhouette": silhouette_score(X_blobs, rotulos),
                   "ari_contra_a_verdade": adjusted_rand_score(y_blobs, rotulos)})

tabela_k = pd.DataFrame(linhas).set_index("k")
display(tabela_k.round(4))

melhor_k = int(tabela_k["silhouette"].idxmax())
rotulos_k = KMeans(n_clusters=melhor_k, n_init=10, random_state=RANDOM_STATE).fit_predict(X_blobs)

fig, eixos = plt.subplots(1, 2, figsize=(12, 4.2))
eixos[0].plot(tabela_k.index, tabela_k["inercia"], marker="o", color="#0F6B68")
eixos[0].set_xlabel("k"); eixos[0].set_ylabel("inércia"); eixos[0].set_title("A inércia sempre cai")
eixos[1].plot(tabela_k.index, tabela_k["silhouette"], marker="o", color="#B45309")
eixos[1].axvline(melhor_k, linestyle="--", linewidth=1)
eixos[1].set_xlabel("k"); eixos[1].set_ylabel("silhouette")
eixos[1].set_title(f"O silhouette tem máximo (aqui, k={melhor_k})")
plt.tight_layout()
plt.show()

### Como ler isso

Repare no formato das duas curvas. A inércia é monótona e achatada no fim: ela sempre diz que mais grupos é melhor, e por isso não decide nada sozinha. O silhouette sobe, tem máximo e volta a cair: a partir do ponto em que se divide um grupo que era de verdade, a distância dentro dos grupos para de melhorar e a separação entre eles piora.

A coluna `ari_contra_a_verdade` está ali só como conferência, e o que ela mostra é reconfortante: no dado fácil o silhouette acertou o número certo de grupos. Em problema real você não tem essa coluna, e é por isso que o próximo passo existe.


## Passo 2: duas meias-luas, onde o formato decide

Agora o conjunto em que o K-Means não pode ganhar: duas meias-luas entrelaçadas. Os dois grupos são de verdade, e nenhuma reta separa um do outro, que é exatamente o que o K-Means sabe desenhar.

O DBSCAN entra com dois parâmetros: `eps`, o raio que define vizinhança, e `min_samples`, quantos vizinhos são necessários para considerar um ponto como parte de uma região densa. Ele não recebe o número de grupos, e devolve -1 para o que considera ruído.


In [ ]:
X_moons, y_moons = make_moons(n_samples=600, noise=0.08, random_state=RANDOM_STATE)
X_moons = StandardScaler().fit_transform(X_moons)

rotulos_kmeans = KMeans(n_clusters=2, n_init=10, random_state=RANDOM_STATE).fit_predict(X_moons)
rotulos_dbscan = DBSCAN(eps=0.3, min_samples=8).fit_predict(X_moons)

linhas = []
for nome, rotulos in {"K-Means (k=2)": rotulos_kmeans, "DBSCAN (eps=0.3)": rotulos_dbscan}.items():
    grupos = len({r for r in rotulos if r != -1})
    linhas.append({"modelo": nome, "grupos": grupos,
                   "ruido": int(np.sum(rotulos == -1)),
                   "silhouette": silhouette_score(X_moons, rotulos) if grupos > 1 else np.nan,
                   "ari_contra_a_verdade": adjusted_rand_score(y_moons, rotulos)})

tabela_moons = pd.DataFrame(linhas).set_index("modelo")
display(tabela_moons.round(4))

fig, eixos = plt.subplots(1, 2, figsize=(12, 4.5))
for eixo, (titulo, rotulos) in zip(eixos, [("K-Means (k=2)", rotulos_kmeans),
                                           ("DBSCAN (eps=0.3)", rotulos_dbscan)]):
    eixo.scatter(X_moons[:, 0], X_moons[:, 1], c=rotulos, cmap="coolwarm", edgecolor="k", s=15)
    eixo.set_title(titulo)
plt.tight_layout()
plt.show()

### Como ler isso

O K-Means corta as meias-luas ao meio por uma reta, e o ARI mostra isso: ele acerta bem menos da metade do que o DBSCAN acerta, mesmo com o número de grupos certo. Não é falta de ajuste, é o desenho do algoritmo: os grupos dele são sempre convexos, e nenhum `k` diferente resolveria.

Agora o detalhe mais importante desta tabela, e que costuma passar batido: o **silhouette do DBSCAN é péssimo**, e o agrupamento dele é quase perfeito contra a verdade, enquanto o do K-Means tem silhouette muito melhor e está errado. É o limite das métricas internas em uma frase: o silhouette mede compacidade, e as meias-luas não são compactas. Métrica interna serve para comparar partições do mesmo formato, não para decidir qual algoritmo está certo.

O DBSCAN acerta, porque a pergunta que ele faz não é "qual centro está mais perto", e sim "este ponto tem vizinhança suficiente para pertencer a uma região densa". Duas consequências práticas: ele encontra grupos de qualquer formato e marca o que está isolado como ruído, o que costuma ser útil (é assim que ele aponta anomalia).

Um detalhe do gráfico que vale a pena notar: o DBSCAN deixou pontos isolados com o rótulo -1. Se você não filtrar isso antes de calcular uma métrica, eles entram como se fossem um grupo, e o número de grupos fica maior do que deveria.

Duas advertências sobre o `eps`: ele depende da escala dos dados, então padronize antes, como está sendo feito aqui; e não existe valor automático. O caminho usual é olhar a distribuição da distância ao `k`-ésimo vizinho e escolher o `eps` no joelho da curva. Experimente `eps=0.15` e `eps=0.5` para ver os dois modos de falha: grupos demais e um grupo só.


## Passo 3: num dado de verdade, onde não existe verdade

Os dígitos do scikit-learn têm 64 colunas e dez classes de verdade que o modelo nunca vê. É o caso realista: você escolhe o número de grupos por argumento externo (dez dígitos) e descobre depois, com os rótulos, se o agrupamento faz sentido.

O ponto deste passo é mostrar que o problema fica muito mais difícil quando as colunas crescem, e é por isso que o próximo notebook da pasta trata de redução de dimensão.


In [ ]:
digitos = load_digits()
X_dig = StandardScaler().fit_transform(digitos.data)

rotulos_dig = KMeans(n_clusters=10, n_init=10, random_state=RANDOM_STATE).fit_predict(X_dig)
print(f"🔎 dígitos: {X_dig.shape[0]} linhas | {X_dig.shape[1]} colunas | 10 grupos pedidos")
print(f"📊 silhouette: {silhouette_score(X_dig, rotulos_dig):.4f}")
print(f"📊 ARI contra os rótulos verdadeiros: {adjusted_rand_score(digitos.target, rotulos_dig):.4f}")

### Como ler isso

O silhouette fica baixo, e o ARI fica longe de 1. Nada disso é defeito do K-Means: em 64 dimensões, todos os pontos ficam aproximadamente à mesma distância uns dos outros, e a noção de "mais perto" perde poder. É a maldição da dimensionalidade, e o sintoma é esse: um agrupamento que parece razoável e não corresponde às classes.

Duas saídas, e as duas passam por reduzir a dimensão antes de agrupar: usar as direções de maior variância (PCA) ou aprender uma representação que preserve vizinhança (t-SNE, autoencoder). Reduzir para 10 ou 20 colunas antes do K-Means costuma melhorar o ARI em dados como este, e é o experimento natural depois deste notebook.

## Resumo

Sem rótulo, a avaliação passa a ser interna: inércia (só compactação, sempre melhora com mais grupos), silhouette (compactação contra separação, tem máximo) e, quando a verdade existe, ARI para conferir. O K-Means é rápido, simples e só enxerga grupos convexos; o DBSCAN aceita qualquer formato, marca ruído e cobra dois parâmetros sensíveis.

A regra prática que fica: padronize antes, escolha o `k` pelo silhouette e nunca pela inércia, e lembre que em alta dimensão o conceito de distância que os dois algoritmos usam fica frágil.

**Próximo passo:** o notebook de redução de dimensão da mesma pasta, que é o pré-processamento mais comum antes de agrupar dado largo.

### Referências

**Os algoritmos**

MacQueen, J. (1967). *Some methods for classification and analysis of multivariate observations.* Proceedings of the Fifth Berkeley Symposium on Mathematical Statistics and Probability.

Ester, M., Kriegel, H.-P., Sander, J., & Xu, X. (1996). *A density-based algorithm for discovering clusters in large spatial databases with noise.* Proceedings of KDD.

**As métricas**

Rousseeuw, P. J. (1987). *Silhouettes: a graphical aid to the interpretation and validation of cluster analysis.* Journal of Computational and Applied Mathematics 20, 53-65.

Hubert, L., & Arabie, P. (1985). *Comparing partitions.* Journal of Classification 2(1), 193-218.

```bibtex
@inproceedings{macqueen1967some,
  title     = {Some methods for classification and analysis of multivariate observations},
  author    = {MacQueen, James},
  booktitle = {Proceedings of the Fifth Berkeley Symposium on Mathematical Statistics and Probability},
  year      = {1967}
}

@inproceedings{ester1996density,
  title     = {A density-based algorithm for discovering clusters in large spatial databases with noise},
  author    = {Ester, Martin and Kriegel, Hans-Peter and Sander, J{\"o}rg and Xu, Xiaowei},
  booktitle = {Proceedings of the Second International Conference on Knowledge Discovery and Data Mining (KDD)},
  year      = {1996}
}

@article{rousseeuw1987silhouettes,
  title   = {Silhouettes: a graphical aid to the interpretation and validation of cluster analysis},
  author  = {Rousseeuw, Peter J.},
  journal = {Journal of Computational and Applied Mathematics},
  volume  = {20},
  pages   = {53--65},
  year    = {1987}
}

@article{hubert1985comparing,
  title   = {Comparing partitions},
  author  = {Hubert, Lawrence and Arabie, Phipps},
  journal = {Journal of Classification},
  volume  = {2},
  number  = {1},
  pages   = {193--218},
  year    = {1985}
}
```
